In [0]:
source_dir = "/Volumes/my_catalog/my_volume/my_file_volume/source_erp/"
schema_dir = "/Volumes/my_catalog/my_volume/my_file_volume/_schemas/cust_az12_bronze/"
checkpoint_dir = "/Volumes/my_catalog/my_volume/my_file_volume/_checkpoints/cust_az12_bronze/"
target_table = "my_catalog.bronze.CUST_AZ12"

df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.schemaLocation", schema_dir)
      .option("cloudFiles.includeExistingFiles", "true")   # first time loads existing file
      .option("header", "true")
      .option("pathGlobFilter", "CUST_AZ12.csv")            # ✅ only this file
      .load(source_dir)
)

(df.writeStream
 .format("delta")
 .option("checkpointLocation", checkpoint_dir)
 .outputMode("append")
 .option("mergeSchema", "true")                             # resolve schema mismatch
 .trigger(once=True)                                       # run once then stop
 .toTable(target_table)
)